In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix
import joblib
import os
import sys

sys.path.append('..')
os.chdir('..')

from src.utils.config_loader import load_config
from src.data_pipeline.preprocess import *
from src.data_pipeline.features import *
from src.models.cf_model import *
from mlops.evaluate import evaluate_model_at_k

config = load_config("configs/data_config.yaml")
print("✅ Imports done")

https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz
✅ Imports done


d:\Ahmed\study\DEPI\tasks\Final_project\recommendation-system\system_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
processed_path = config['paths']['processed_data']

# 2. Load Processed DataFrames
print("[INFO] Loading DataFrames...")
train_df = pd.read_parquet(processed_path + 'train.parquet')
val_df   = pd.read_parquet(processed_path + 'val.parquet')

# 3. Load Encoders and Scaler
print("[INFO] Loading Encoders and Scaler...")
encoders = joblib.load(processed_path + 'encoders.pkl')
scaler   = joblib.load(processed_path + 'scaler.pkl')

# 4. Load the Fixed-Shape Sparse Matrices
print("[INFO] Loading Sparse Matrices...")
train_matrix = joblib.load(processed_path + 'train_matrix.pkl')
val_matrix   = joblib.load(processed_path + 'val_matrix.pkl')

print("\n=== Data Overview ===")
print(f"Train DF shape:      {train_df.shape}")
print(f"Validation DF shape: {val_df.shape}")
print(f"Train Matrix shape:  {train_matrix.shape}")

# Preview the first few rows
train_df.head()

[INFO] Loading DataFrames...
[INFO] Loading Encoders and Scaler...
[INFO] Loading Sparse Matrices...

=== Data Overview ===
Train DF shape:      (53908, 14)
Validation DF shape: (4677, 14)
Train Matrix shape:  (7468, 4218)


,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,interaction_weight,user_segment,user_verified_ratio,item_avg_rating,is_weekend
0,5.0,Very Good Product,None,B00L3KNY22,983,0,2016-06-20 17:38:33,0.0,1,1.0,Medium,1.0,5.000000,0
1,4.0,The Lumina Phone was like $5. 00 monthly to bu...,"I have a desk top, windows 10, Lumina Phone wi...",B01AHBAJU2,1541,0,2016-12-03 18:44:12,0.0,1,1.0,Medium,1.0,4.769231,1
2,5.0,Great,Installed as explained. Watched video. Looks ...,B01LX5UXYU,1748,0,2016-12-08 11:13:56,0.0,1,1.0,Medium,1.0,4.833333,0
3,5.0,Excellent Fit and Quality,The best screen protector I have installed. w...,B0144AAIWQ,1409,0,2017-03-24 20:55:10,0.0,1,1.0,Medium,1.0,4.952381,0
4,5.0,San Dusk,None,B013TMNKAW,3047,0,2017-03-27 11:13:55,0.0,1,1.0,Medium,1.0,4.910256,0


In [3]:
# Extract the item encoder to decode product IDs
item_encoder = encoders['parent_asin']

# 1. Setup Scenario 1: The First Item (Sparse / Rare)
test_item_idx = 0
test_asin = item_encoder.inverse_transform([test_item_idx])[0]

# 2. Setup Scenario 2: The Most Popular Item (Dense / Frequent)
# Summing interactions across all users for each item
item_interaction_counts = np.asarray(train_matrix.sum(axis=0)).flatten()
popular_item_idx = item_interaction_counts.argmax()
popular_asin = item_encoder.inverse_transform([popular_item_idx])[0]

print(f"[INFO] First Item ASIN: {test_asin} (Interactions: {item_interaction_counts[test_item_idx]})")
print(f"[INFO] Popular Item ASIN: {popular_asin} (Interactions: {item_interaction_counts[popular_item_idx]})")

# 3. Helper Function for Comparison
def print_model_comparison(model, item_idx, asin, model_name):
    recs = model.recommend(item_id=item_idx, n_recommendations=5)
    print(f"\n--- {model_name} Recommendations for {asin} ---")
    for rank, (idx, score) in enumerate(recs, start=1):
        rec_asin = item_encoder.inverse_transform([idx])[0]
        # Adjust score calculation for KNN (distance to similarity)
        if model_name == "Item-Based KNN":
            score = 1 - score 
        print(f"Rank {rank}: ASIN {rec_asin} | Score: {score:.4f}")

[INFO] First Item ASIN: 1426320965 (Interactions: 30.0)
[INFO] Popular Item ASIN: B01K8B8YA8 (Interactions: 2726.0)


In [4]:
# 1. KNN Model
print("\n[INFO] 1. Training Item-Based KNN.")
knn_model = ItemBasedKNN(n_neighbors=10, metric='cosine')
knn_model.fit(train_matrix)

# 2. ALS Model
print("\n[INFO] 2. Training ALS Model.")
als_model = ALSRecommender(factors=64, iterations=20)
als_model.fit(train_matrix)

# 3. BPR Model
print("\n[INFO] 3. Training BPR Model.")
bpr_model = BPRRecommender(factors=64, iterations=100)
bpr_model.fit(train_matrix)

# 4. SVD Model
print("\n[INFO] 4. Training SVD Model.")
svd_model = SVDRecommender(n_components=64)
svd_model.fit(train_matrix)

print("\n[INFO] All 4 Models successfully trained and loaded in memory.")


[INFO] 1. Training Item-Based KNN.
[INFO] Fitting Item-Based KNN with metric='cosine'...
[INFO] KNN Model fitting complete. ✅

[INFO] 2. Training ALS Model.


d:\Ahmed\study\DEPI\tasks\Final_project\recommendation-system\system_env\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


[INFO] Fitting ALS Model with 64 latent factors...


100%|██████████| 20/20 [00:00<00:00, 48.98it/s]


[INFO] ALS Model fitting complete. 

[INFO] 3. Training BPR Model.
[INFO] Fitting BPR Model with 64 latent factors...


100%|██████████| 100/100 [00:00<00:00, 221.08it/s, train_auc=75.64%, skipped=0.88%]


[INFO] BPR Model fitting complete. ✅

[INFO] 4. Training SVD Model.
[INFO] Fitting SVD Model with 64 components...
[INFO] SVD Model fitting complete. ✅

[INFO] All 4 Models successfully trained and loaded in memory.


In [5]:

print("="*80)
print(f"========== SCENARIO 1: First Item (ASIN: {test_asin}) ==========")
print("="*80)

print_model_comparison(knn_model, test_item_idx, test_asin, "Item-Based KNN")
print_model_comparison(als_model, test_item_idx, test_asin, "ALS")
print_model_comparison(bpr_model, test_item_idx, test_asin, "BPR")
print_model_comparison(svd_model, test_item_idx, test_asin, "SVD")

========== SCENARIO 1: First Item (ASIN: 1426320965) ==========

--- Item-Based KNN Recommendations for 1426320965 ---
Rank 1: ASIN B076HFYGLR | Score: 0.2513
Rank 2: ASIN B09YGYC783 | Score: 0.2140
Rank 3: ASIN B0BVW43LNY | Score: 0.1895
Rank 4: ASIN B09682TWD8 | Score: 0.1719
Rank 5: ASIN B01D5NR70E | Score: 0.1667

--- ALS Recommendations for 1426320965 ---
Rank 1: ASIN B07C9H4JY8 | Score: 0.5472
Rank 2: ASIN B07MB92NP5 | Score: 0.4642
Rank 3: ASIN B00IOWB438 | Score: 0.4525
Rank 4: ASIN B08YQJDG6F | Score: 0.4439
Rank 5: ASIN B0BS6KLLT1 | Score: 0.4350

--- BPR Recommendations for 1426320965 ---
Rank 1: ASIN B07N8VFFNS | Score: 0.6961
Rank 2: ASIN B0144AAIWQ | Score: 0.6612
Rank 3: ASIN B0189XYY0Q | Score: 0.6434
Rank 4: ASIN B07S4MFN8Z | Score: 0.6327
Rank 5: ASIN B07C9H4JY8 | Score: 0.6271

--- SVD Recommendations for 1426320965 ---
Rank 1: ASIN B08YQJDG6F | Score: 0.6319
Rank 2: ASIN B0051FWA7U | Score: 0.5007
Rank 3: ASIN B0077BB3VA | Score: 0.4898
Rank 4: ASIN B071DWG4C8 | Sco

In [6]:
print("="*80)
print(f"========== SCENARIO 2: Most Popular Item (ASIN: {popular_asin}) ==========")
print("="*80)

print_model_comparison(knn_model, popular_item_idx, popular_asin, "Item-Based KNN")
print_model_comparison(als_model, popular_item_idx, popular_asin, "ALS")
print_model_comparison(bpr_model, popular_item_idx, popular_asin, "BPR")
print_model_comparison(svd_model, popular_item_idx, popular_asin, "SVD")

========== SCENARIO 2: Most Popular Item (ASIN: B01K8B8YA8) ==========

--- Item-Based KNN Recommendations for B01K8B8YA8 ---
Rank 1: ASIN B07456BG8N | Score: 0.1573
Rank 2: ASIN B075X8471B | Score: 0.1569
Rank 3: ASIN B07KTYJ769 | Score: 0.1147
Rank 4: ASIN B010BWYDYA | Score: 0.1117
Rank 5: ASIN B07HZLHPKP | Score: 0.1077

--- ALS Recommendations for B01K8B8YA8 ---
Rank 1: ASIN B0C38JGJW3 | Score: 0.5750
Rank 2: ASIN B01K9KW576 | Score: 0.5070
Rank 3: ASIN B0765XK48L | Score: 0.4937
Rank 4: ASIN B079FXKHY9 | Score: 0.4891
Rank 5: ASIN B08C3YBBHM | Score: 0.4853

--- BPR Recommendations for B01K8B8YA8 ---
Rank 1: ASIN B0BX8DMYWL | Score: 0.9070
Rank 2: ASIN B01KIOU214 | Score: 0.9035
Rank 3: ASIN B01IQEJBNS | Score: 0.9014
Rank 4: ASIN B01K9KW576 | Score: 0.8777
Rank 5: ASIN B081498PMW | Score: 0.8767

--- SVD Recommendations for B01K8B8YA8 ---
Rank 1: ASIN B0C38JGJW3 | Score: 0.8241
Rank 2: ASIN B08C3YBBHM | Score: 0.7788
Rank 3: ASIN B079FXKHY9 | Score: 0.5578
Rank 4: ASIN B07G3HYDJ

In [7]:
# Define the save path
model_save_path = config['paths']['processed_data']  # Or create a config['paths']['models']

print("[INFO] Saving Models to disk.")

joblib.dump(knn_model, os.path.join(model_save_path, 'knn_baseline.pkl'))
joblib.dump(als_model, os.path.join(model_save_path, 'als_model.pkl'))
joblib.dump(bpr_model, os.path.join(model_save_path, 'bpr_model.pkl'))
joblib.dump(svd_model, os.path.join(model_save_path, 'svd_model.pkl'))

print("[INFO] All models successfully saved")

[INFO] Saving Models to disk.
[INFO] All models successfully saved


In [8]:
# Define the load path (same as save path)
model_load_path = config['paths']['processed_data'] 

print("[INFO] Loading Models from disk...")

knn_model = joblib.load(os.path.join(model_load_path, 'knn_baseline.pkl'))
als_model = joblib.load(os.path.join(model_load_path, 'als_model.pkl'))
bpr_model = joblib.load(os.path.join(model_load_path, 'bpr_model.pkl'))
svd_model = joblib.load(os.path.join(model_load_path, 'svd_model.pkl'))

print("[INFO] All models successfully loaded into memory.")

[INFO] Loading Models from disk...
[INFO] All models successfully loaded into memory.


In [9]:
K = 10
SAMPLE_USERS = 500

valid_users = np.where(val_matrix.getnnz(axis=1) > 0)[0]

np.random.seed(42) 
users_to_evaluate = np.random.choice(
    valid_users, 
    size=min(SAMPLE_USERS, len(valid_users)), 
    replace=False
)

print(f"[INFO] Evaluating all models on {len(users_to_evaluate)} valid users...\n")

print("="*60)
print(f"========== EVALUATING ALS MODEL ==========")
print("="*60)
als_metrics = evaluate_model_at_k(als_model, val_matrix, users_to_evaluate, k=K)
print(f"ALS Results: {als_metrics}\n")

print("="*60)
print(f"========== EVALUATING BPR MODEL ==========")
print("="*60)
bpr_metrics = evaluate_model_at_k(bpr_model, val_matrix, users_to_evaluate, k=K)
print(f"BPR Results: {bpr_metrics}\n")

print("="*60)
print(f"========== EVALUATING SVD MODEL ==========")
print("="*60)
svd_metrics = evaluate_model_at_k(svd_model, val_matrix, users_to_evaluate, k=K)
print(f"SVD Results: {svd_metrics}\n")

print("="*60)
print(f"========== EVALUATING KNN MODEL ==========")
print("="*60)
knn_metrics = evaluate_model_at_k(knn_model, val_matrix, users_to_evaluate, k=K)
print(f"KNN Results: {knn_metrics}\n")

[INFO] Evaluating all models on 500 valid users...

========== EVALUATING ALS MODEL ==========


Evaluating Users: 100%|██████████| 500/500 [00:00<00:00, 4539.92it/s]


ALS Results: {'HitRate@10': np.float64(0.036), 'Precision@10': np.float64(0.0036), 'Adj_Precision@10': np.float64(0.0187), 'Recall@10': np.float64(0.0187), 'MRR@10': np.float64(0.0133), 'NDCG@10': np.float64(0.0113)}

========== EVALUATING BPR MODEL ==========


Evaluating Users: 100%|██████████| 500/500 [00:00<00:00, 4449.05it/s]


BPR Results: {'HitRate@10': np.float64(0.028), 'Precision@10': np.float64(0.003), 'Adj_Precision@10': np.float64(0.0143), 'Recall@10': np.float64(0.0143), 'MRR@10': np.float64(0.0062), 'NDCG@10': np.float64(0.007)}

========== EVALUATING SVD MODEL ==========


Evaluating Users: 100%|██████████| 500/500 [00:00<00:00, 3620.13it/s]


SVD Results: {'HitRate@10': np.float64(0.02), 'Precision@10': np.float64(0.002), 'Adj_Precision@10': np.float64(0.009), 'Recall@10': np.float64(0.009), 'MRR@10': np.float64(0.0073), 'NDCG@10': np.float64(0.0059)}

========== EVALUATING KNN MODEL ==========


Evaluating Users: 100%|██████████| 500/500 [01:42<00:00,  4.87it/s]

KNN Results: {'HitRate@10': np.float64(0.014), 'Precision@10': np.float64(0.0014), 'Adj_Precision@10': np.float64(0.0099), 'Recall@10': np.float64(0.0099), 'MRR@10': np.float64(0.0023), 'NDCG@10': np.float64(0.0038)}

